# Day 11 - GANs & VAEs: Image Augmentation

## RetailMind AI Increment

Day 11 addresses class imbalance in the Phase-2 catalog-vision pipeline. The implementation selects the least represented product category, trains a convolutional **variational autoencoder (VAE)** on that category, rejects weak or near-duplicate samples, and defines a controlled before/after classifier evaluation.

### Business objective
Rare catalog categories often receive lower recall because the classifier sees fewer examples. Synthetic augmentation can reduce this exposure gap, but only when generated images are sufficiently varied and realistic. The project therefore separates generation, quality filtering, and downstream fairness evaluation.

## Why a VAE was selected

A VAE is used instead of a GAN for this increment because the small CPU-friendly model provides stable training, explicit latent sampling, and a reconstruction objective that is easier to monitor. The trade-off is that VAE images may be smoother than GAN outputs. Synthetic images are never automatically trusted: a variation threshold and cosine-similarity gate reject flat outputs and likely duplicates.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day11_vae_image_augmentation as day11

print("TensorFlow:", day11.tf.__version__)
print("Image size:", day11.IMAGE_SIZE)
print("Latent dimension:", day11.LATENT_DIM)

## 1. Identify the under-represented category

Expected layout:

```text
data/fashion_products/
|-- Apparel/
|-- Footwear/
|-- Accessories/
`-- ...
```

The category with the fewest valid JPG/JPEG/PNG files is selected. In a production audit, this count should be compared with the Day-6/7 confusion matrix so the selected category is both thin and operationally weak.

In [ ]:
counts, rare_category = day11.discover_categories(repo_root / day11.DATA_DIR)
print("Category counts:")
for category, count in sorted(counts.items(), key=lambda item: item[1]):
    print(f"{category:25s} {count:6d}")
print("Selected rare category:", rare_category)

## 2. Load and normalize rare-category images

Only the selected category is used for VAE training. Images are resized to 64×64 RGB and scaled to `[0, 1]`. This keeps CPU training practical while preserving enough shape and color information for augmentation review.

In [ ]:
rare_ds = day11.load_rare_dataset(rare_category, repo_root / day11.DATA_DIR)
first_batch = next(iter(rare_ds))
print("Batch shape:", first_batch.shape)
print("Value range:", float(day11.tf.reduce_min(first_batch)), float(day11.tf.reduce_max(first_batch)))

## 3. Build the convolutional VAE

The encoder reduces each image through three strided convolution blocks and produces `z_mean` and `z_log_var`. The reparameterization trick samples a 64-dimensional latent code. The decoder reconstructs the image through transposed convolutions. Training minimizes reconstruction error plus a small KL-divergence term.

In [ ]:
encoder = day11.build_encoder()
decoder = day11.build_decoder()
vae = day11.VAE(encoder, decoder, name="day11_rare_category_vae")
vae.compile(optimizer=day11.keras.optimizers.Adam(1e-3))
encoder.summary()
decoder.summary()

## 4. Train the VAE

The default is 15 epochs. Reconstruction loss should fall over time while KL loss remains non-zero. A KL value collapsing to almost zero can indicate that the decoder is ignoring the latent code, while nearly identical generated samples may indicate poor latent coverage.

In [ ]:
history = vae.fit(rare_ds, epochs=day11.VAE_EPOCHS, verbose=2)
print("Final reconstruction loss:", history.history["reconstruction"][-1])
print("Final KL loss:", history.history["kl"][-1])

## 5. Generate samples and apply quality gates

The target is to reduce the count gap between the rare category and the largest category. The quality gate rejects:

- low-variation images that are nearly blank;
- candidates with cosine similarity above 0.985 to another generated candidate.

These automated checks are preliminary. A human catalog-quality review should still inspect product shape, texture, clipping, background artifacts, and category consistency.

In [ ]:
target_count = max(counts.values())
requested = max(0, target_count - counts[rare_category])
print("Largest category count:", target_count)
print("Rare category count:", counts[rare_category])
print("Requested synthetic images:", requested)

# Uncomment to generate files.
# kept, attempted = day11.generate_and_filter(decoder, requested, repo_root / day11.GENERATED_DIR)
# print("Kept:", kept, "Attempted:", attempted)

## 6. Inspect diversity and mode-collapse risk

Mode collapse is primarily associated with GANs, but a VAE can still produce low-diversity outputs. Review multiple latent samples rather than one attractive example. Useful indicators include acceptance rate, pairwise similarity, color/shape diversity, and whether generated items remain recognizably within the intended product class.

In [ ]:
sample_images = decoder(day11.tf.random.normal((16, day11.LATENT_DIM)), training=False)
quality = day11.image_quality_mask(sample_images)
flat = day11.tf.reshape(sample_images, (16, -1))
similarity = day11.tf.matmul(
    day11.tf.math.l2_normalize(flat, axis=1),
    day11.tf.math.l2_normalize(flat, axis=1),
    transpose_b=True,
)
print("Accepted in sample batch:", int(day11.tf.reduce_sum(day11.tf.cast(quality, day11.tf.int32))))
print("Average off-diagonal similarity:", float((day11.tf.reduce_sum(similarity)-16)/(16*15)))

## 7. Controlled classifier comparison

To measure real value, train the **same classifier architecture, initialization policy, split, epochs, and validation set** twice:

1. Original training folders only.
2. A copied training tree with accepted synthetic images added only to the rare-category folder.

The untouched validation set must contain real images only. Report overall accuracy, rare-class accuracy, macro average per-class accuracy, and the minimum per-class accuracy. A fairness claim requires improvement in the weak class without unacceptable degradation elsewhere.

In [ ]:
# Baseline example:
# train_original, validation, class_names = day11.classifier_datasets(repo_root / day11.DATA_DIR)
# baseline = day11.build_classifier(len(class_names))
# baseline.fit(train_original, validation_data=validation,
#              epochs=day11.CLASSIFIER_EPOCHS, verbose=2)
# before = day11.per_class_accuracy(baseline, validation, class_names)
#
# Repeat with data/fashion_products_augmented, then call:
# day11.save_comparison(class_names, before, after,
#                       repo_root / "models/day11_per_class_accuracy.csv")

## 8. Weekly Assessment 04 analysis template

Document results only after executing the controlled experiment:

- Rare category and original/augmented training counts.
- VAE reconstruction and KL behavior.
- Candidate acceptance rate after quality filtering.
- Rare-class accuracy before and after augmentation.
- Macro per-class accuracy and minimum-class accuracy lift.
- Any class harmed by augmentation.
- Visual defects, repetition, or category leakage found during review.

Synthetic augmentation is successful only if real holdout performance improves. Attractive generated images alone are not evidence of classifier or fairness benefit.

## 9. Integration and next step

Accepted images can be incorporated into the Day-6/7 catalog-classification pipeline through a separate augmented training directory, preserving the original dataset for reproducibility. The VAE encoder and decoder are saved independently for later sampling and audit. Day 12 begins text intelligence with review cleaning and a classical sentiment baseline.